In [1]:
!pip install -U langchain langchain-community langchain-core langchain-qdrant qdrant-client langchain-huggingface 
!pip install fastembed sentence-transformers huggingface-hub>=0.36.2 
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 

# Test Qdrant

In [2]:
import json
import re
from tqdm import tqdm

parent_input_file = "/kaggle/input/datasets/haivuu/chunking-v5/law_data_final.jsonl"
child_input_file = "/kaggle/input/datasets/haivuu/chunking-v5/law_data_khoan_v3.jsonl"
output_file = "/kaggle/working/law_data_khoan_enriched.jsonl"

# =====================================================================
# BƯỚC 1: KHAI QUẬT VÀ XÂY DỰNG TỪ ĐIỂN
# =====================================================================
print(" BƯỚC 1: Xây dựng Từ điển Ngữ cảnh từ file ĐIỀU...")
context_dict = {}

with open(parent_input_file, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Quét file Điều"):
        if not line.strip(): continue
        data = json.loads(line)
        meta = data.get("metadata", {})
        
        doc_id = meta.get("doc_id")
        article_num = meta.get("article_number")
        
        if not article_num:
            for i in range(1, 7):
                ctx = meta.get(f"context_lv{i}", "")
                if ctx:
                    match = re.search(r"Điều\s+(\d+[a-zA-Z]*)", ctx, re.IGNORECASE)
                    if match:
                        article_num = match.group(1)
                        break
                        
        if not article_num:
            original_text = data.get("text", "")
            match = re.search(r"Điều\s+(\d+[a-zA-Z]*)", original_text, re.IGNORECASE)
            if match:
                article_num = match.group(1)

        if doc_id and article_num:
            parent_id = f"{doc_id}_Dieu_{article_num}"
            
            contexts = []
            for i in range(1, 7):
                ctx = meta.get(f"context_lv{i}", "").strip()
                if ctx:
                    contexts.append(ctx)
            
            if contexts:
                rich_context = " > ".join(contexts)
            else:
                text_content = data.get("text", "")
                if "\n---\n" in text_content:
                    text_content = text_content.split("\n---\n")[-1].strip()
                rich_context = text_content.split("\n")[0].strip()
                
            context_dict[parent_id] = rich_context

print(f" Cứu thành công {len(context_dict)} Tiêu đề Điều luật vào RAM!")

# =====================================================================
# BƯỚC 2: TIÊM NGỮ CẢNH VÀO FILE KHOẢN (ĐÃ FIX LỖI TÊN VĂN BẢN)
# =====================================================================
print("\n BƯỚC 2: Tiêm ngữ cảnh vào file KHOẢN...")
processed = 0
orphans = 0

with open(child_input_file, "r", encoding="utf-8") as fin, \
     open(output_file, "w", encoding="utf-8") as fout:
    
    for line in tqdm(fin, desc="Tiêm ngữ cảnh file Khoản"):
        if not line.strip(): continue
        data = json.loads(line)
        meta = data.get("metadata", {})
        original_text = data.get("text", "")
        
        if "\n---\n" in original_text:
            clean_content = original_text.split("\n---\n")[-1].strip()
        else:
            clean_content = original_text.strip()
            
        doc_id = meta.get("doc_id")
        article_num = meta.get("article_number")
        clause_num = meta.get("clause_number")
        
        ten_goc = meta.get("ten_van_ban", "Không rõ tên")
        so_hieu = meta.get("so_hieu", "")
        ten_van_ban_day_du = f"{ten_goc} ({so_hieu})" if so_hieu else ten_goc
        
        if doc_id and article_num:
            parent_id = f"{doc_id}_Dieu_{article_num}"
            rich_context = context_dict.get(parent_id, f"Điều {article_num}")
            khoan_str = f"Khoản {clause_num}" if clause_num else "Nội dung"
            
            # Text được lắp ráp hoàn chỉnh
            semantic_text = f"[{ten_van_ban_day_du}] - {rich_context} - {khoan_str}: {clean_content}"
            data["text"] = semantic_text
            processed += 1
        else:
            data["text"] = f"[{ten_van_ban_day_du}]: {clean_content}"
            orphans += 1
            
        fout.write(json.dumps(data, ensure_ascii=False) + "\n")

print(f"\n HOÀN TẤT. Đã xử lý: {processed} | Mồ côi: {orphans}")

⚙️ BƯỚC 1: Xây dựng Từ điển Ngữ cảnh từ file ĐIỀU...


Quét file Điều: 48236it [00:07, 6440.45it/s]


✅ ĐÃ FIX LỖI: Cứu thành công 44539 Tiêu đề Điều luật vào RAM!

⚙️ BƯỚC 2: Tiêm ngữ cảnh vào file KHOẢN...


Tiêm ngữ cảnh file Khoản: 213703it [00:14, 14779.02it/s]


🎉 HOÀN TẤT BƯỚC 2! Đã xử lý: 194336 | Mồ côi: 19367


In [3]:
import os
import pandas as pd
from tqdm.auto import tqdm
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# =====================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# =====================================================================
ENRICHED_FILE = "/kaggle/working/law_data_khoan_enriched.jsonl"
NEW_DB_PATH = "/kaggle/working/qdrant_v6"

# =====================================================================
# 2. LOAD VÀ LÀM SẠCH DỮ LIỆU
# =====================================================================
print("🧹 BƯỚC 1: Đang tải và dọn dẹp rác tiếng Anh từ file Enriched...")
df = pd.read_json(ENRICHED_FILE, lines=True)

# Lọc bỏ các văn bản dịch tiếng Anh
eng_pattern = 'Law No|Decree No|Circular No|Resolution No|Decision No|dated'
mask = df['text'].str.contains(eng_pattern, case=False, na=False)
df_clean = df[~mask].copy()

# Cắt bỏ các chunk đột biến > 5000 ký tự để chống tràn RAM và Context Window
df_clean = df_clean[df_clean['text'].str.len() <= 5000]

print(f" Đã dọn xong! Tổng số vector vàng nguyên chất sẽ nhúng: {len(df_clean):,}")

# =====================================================================
# 3. CHUẨN BỊ DOCUMENTS
# =====================================================================
print("\n BƯỚC 2: Chuẩn bị dữ liệu để nhúng...")
documents = []
for _, row in df_clean.iterrows():
    # Giữ lại doc_id và ten_van_ban làm metadata để in nguồn cho đẹp
    metadata = {
        "doc_id": str(row.get("metadata", {}).get("doc_id", "")),
        "ten_van_ban": str(row.get("metadata", {}).get("ten_van_ban", ""))
    }
    documents.append(Document(page_content=row["text"], metadata=metadata))

# =====================================================================
# 4. KHỞI TẠO EMBEDDING VÀ TẠO COLLECTION
# =====================================================================
print("\n BƯỚC 3: Đang tải AI Embeddings...")
dense_embeddings = HuggingFaceEmbeddings(
    model_name="bkai-foundation-models/vietnamese-bi-encoder",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

# Đảm bảo thư mục rỗng và không bị kẹt lock
os.makedirs(NEW_DB_PATH, exist_ok=True)
lock_file = os.path.join(NEW_DB_PATH, ".lock")
if os.path.exists(lock_file): os.remove(lock_file)

BATCH_SIZE = 2000
first_batch = documents[:BATCH_SIZE]

print(f" BƯỚC 4: Đang xây dựng Collection Hybrid và nhúng {BATCH_SIZE} chunks đầu tiên...")
qdrant_db = QdrantVectorStore.from_documents(
    documents=first_batch,
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    collection_name="legal_enriched_chunks",
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse",
    path=NEW_DB_PATH  
)

# =====================================================================
# 5. NHÚNG CÁC LÔ CÒN LẠI
# =====================================================================


for i in tqdm(range(BATCH_SIZE, len(documents), BATCH_SIZE), desc="Đang nhúng Vectors"):
    batch = documents[i : i + BATCH_SIZE]
    qdrant_db.add_documents(batch)


🧹 BƯỚC 1: Đang tải và dọn dẹp rác tiếng Anh từ file Enriched...
✅ Đã dọn xong! Tổng số vector vàng nguyên chất sẽ nhúng: 206,133

📝 BƯỚC 2: Chuẩn bị dữ liệu để nhúng...

🧠 BƯỚC 3: Đang tải AI Embeddings...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

🛠️ BƯỚC 4: Đang xây dựng Collection Hybrid và nhúng 2000 chunks đầu tiên...

🚀 BƯỚC 5: Bắt đầu nhúng các Lô còn lại (Pha này mất khoảng 30p - 1 tiếng)...


Đang nhúng Vectors:   0%|          | 0/103 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/langchain_qdrant/qdrant.py:513: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20064 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self.client.upsert(



🎉 HOÀN TẤT! Hệ thống CSDL Qdrant V6 Enriched đã sẵn sàng!


In [5]:
import shutil
import os
from IPython.display import FileLink, display

# --- CẤU HÌNH ĐƯỜNG DẪN ---
DB_FOLDER = "/kaggle/working/qdrant_v6"         # Thư mục gốc chứa DB của bạn
ZIP_FILE_PATH = "/kaggle/working/qdrant_kaggle" # Tên file nén (không cần ghi đuôi .zip)

print(" Đang tiến hành nén toàn bộ Database. Quá trình này có thể mất 1-2 phút tùy dung lượng...")

# Nén thư mục thành file .zip
shutil.make_archive(ZIP_FILE_PATH, 'zip', DB_FOLDER)

print(f" Đã nén thành công! Kích thước file: {os.path.getsize(ZIP_FILE_PATH + '.zip') / (1024*1024):.2f} MB")
print(" BẤM VÀO ĐƯỜNG LINK BÊN DƯỚI ĐỂ TẢI VỀ MÁY TÍNH 👇")

# Hiển thị link tải trực tiếp ngay trong Kaggle
display(FileLink(r'qdrant_kaggle.zip'))

📦 Đang tiến hành nén toàn bộ Database. Quá trình này có thể mất 1-2 phút tùy dung lượng...
✅ Đã nén thành công! Kích thước file: 825.15 MB
👇 BẤM VÀO ĐƯỜNG LINK BÊN DƯỚI ĐỂ TẢI VỀ MÁY TÍNH 👇


/kaggle/working/qdrant_kaggle.zip

In [ ]:
aaa